<a href="https://colab.research.google.com/github/TheAlishbahWaheed/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TheAlishbahWaheed/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Setup — repo root + data
import os
import numpy as np
import pandas as pd

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.exists("flyrank-ml-internship"):
        !git clone https://github.com/TheAlishbahWaheed/flyrank-ml-internship.git
    os.chdir("flyrank-ml-internship")

RANDOM_STATE = 42

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
print("Shape:", df.shape, "| base decline rate:", round(df["is_declining_label"].mean(), 3))
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 166, done.
remote: Counting objects: 100% (166/166), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 166 (delta 60), reused 93 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (166/166), 1.87 MiB | 9.95 MiB/s, done.
Resolving deltas: 100% (60/60), done.
Shape: (30000, 45) | base decline rate: 0.542


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My lane (Refresh / Content Opportunity Scoring) is a **ranking** problem — *which pages first?*
— evaluated with **precision@K**, same as Week 4. Per the toolkit table, a ranking question
wants a classifier's probability score, not a hard label, so I'm predicting `is_declining_label`
(yes/no, observed from `trend_direction`) and ranking by predicted probability.

**Method 1 — Logistic Regression.** Readable start: a coefficient per feature, signs a human
can sanity-check. Fits "yes/no with an observed label."

**Method 2 — Random Forest.** Stronger candidate, handles non-linear/interaction effects (e.g.
the non-monotonic staleness and impression-tier patterns the Week-4 signal audit already found)
without me hand-engineering interactions. Depth and leaf size are kept modest (`max_depth=8`,
`min_samples_leaf=20`) on purpose — this is 30k rows across 32 clients, and an unconstrained
forest would just memorize per-client quirks.

**Feature set — one correction from the Week-3 data contract.** That contract listed
`impressions_last_30d` / `impressions_prev_30d` as features. They can't be: `trend_pct` (the
direct source of the label) is *literally* `(last_30d − prev_30d) / prev_30d`, so those two
columns would hand the model its own answer. I use the pipeline's own canonical safe list
instead (`scripts/ml_utils.py: MODEL_NUMERIC_FEATURES` / `MODEL_CATEGORICAL_FEATURES`) —
90-day totals (logged, since they're heavy-tailed), current-state rates and tiers, lifecycle
fields, and keyword/content metadata. No `trend_direction`, `trend_pct`, `impressions_last_30d`,
or `impressions_prev_30d` anywhere in the feature set. `content_id` / `client_id` are used only
for grouping, never as features.

In [2]:
# Leakage-safe feature set (matches scripts/ml_utils.py's canonical MODEL_*_FEATURES,
# with impressions_last_30d / impressions_prev_30d correctly excluded as label-adjacent)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

# Missingness follows content_type (data dictionary) -> add has_-flags before filling,
# so a blind fillna(0) doesn't quietly become a content-type signal.
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_scroll_data"] = df["scroll_rate"].notna().astype(int)

NUM_FEATS = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "has_keyword_data", "has_word_count", "has_scroll_data",
]
CAT_FEATS = [
    "competition_level", "content_type", "main_intent", "age_tier", "freshness_tier",
    "word_count_tier", "impression_tier", "position_tier",
]

for c in ["search_volume", "competition", "cpc", "word_count", "char_count", "scroll_rate"]:
    df[c] = df[c].fillna(0)
for c in CAT_FEATS:
    df[c] = df[c].fillna("unknown").astype(str)

excluded = ["trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d",
            "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d",
            "is_declining_label", "content_id", "client_id"]
print("Features used:", len(NUM_FEATS) + len(CAT_FEATS), "| explicitly excluded (label-adjacent or IDs):", excluded)
print("Any NaN left in features:", int(df[NUM_FEATS].isna().sum().sum()))

Features used: 29 | explicitly excluded (label-adjacent or IDs): ['trend_direction', 'trend_pct', 'impressions_last_30d', 'impressions_prev_30d', 'clicks_last_30d', 'clicks_prev_30d', 'sessions_last_30d', 'sessions_prev_30d', 'is_declining_label', 'content_id', 'client_id']
Any NaN left in features: 0


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
from sklearn.model_selection import GroupShuffleSplit

X = df[NUM_FEATS + CAT_FEATS]
y = df["is_declining_label"].values
groups = df["client_id"].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(df, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])
print("Train rows:", len(train_idx), "| Test rows:", len(test_idx))
print("Train clients:", len(train_clients), "| Test clients:", len(test_clients))
print("Client overlap between train and test (must be 0):", len(train_clients & test_clients))
print("Train decline rate:", round(y_train.mean(), 3), "| Test decline rate:", round(y_test.mean(), 3))
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Train rows: 22885 | Test rows: 7115
Train clients: 24 | Test clients: 8
Client overlap between train and test (must be 0): 0
Train decline rate: 0.55 | Test decline rate: 0.517


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score

# Week-4 baseline, recomputed identically, restricted to the test split
stale = (df["freshness_tier"] == "91-180").astype(int)
visible = df["impression_tier"].isin(["moderate", "good"]).astype(int)
df["baseline_score"] = stale * visible * df["impressions_90d"]
baseline_test_scores = df.iloc[test_idx]["baseline_score"].values

# Logistic Regression
lr_pre = ColumnTransformer([
    ("num", StandardScaler(), NUM_FEATS),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATS),
])
logreg = Pipeline([("pre", lr_pre), ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])
logreg.fit(X_train, y_train)
logreg_proba = logreg.predict_proba(X_test)[:, 1]

# Random Forest
rf_pre = ColumnTransformer([
    ("num", "passthrough", NUM_FEATS),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATS),
])
rf = Pipeline([("pre", rf_pre), ("clf", RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=RANDOM_STATE, n_jobs=-1))])
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(y_true)[order[:k]].mean())

base_rate = y_test.mean()
rows = []
for k in [10, 20, 50, 100, 200]:
    rows.append({
        "k": k,
        "base_rate": round(base_rate, 3),
        "baseline_precision@k": round(precision_at_k(y_test, baseline_test_scores, k), 3),
        "logreg_precision@k": round(precision_at_k(y_test, logreg_proba, k), 3),
        "rf_precision@k": round(precision_at_k(y_test, rf_proba, k), 3),
    })
comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))

print("\nROC-AUC (full ranking) — baseline:", round(roc_auc_score(y_test, baseline_test_scores), 3),
      "| logreg:", round(roc_auc_score(y_test, logreg_proba), 3),
      "| rf:", round(roc_auc_score(y_test, rf_proba), 3))
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


  k  base_rate  baseline_precision@k  logreg_precision@k  rf_precision@k
 10      0.517                  0.60                0.90           0.400
 20      0.517                  0.45                0.80           0.550
 50      0.517                  0.38                0.74           0.540
100      0.517                  0.32                0.71           0.560
200      0.517                  0.44                0.67           0.565

ROC-AUC (full ranking) — baseline: 0.492 | logreg: 0.61 | rf: 0.603


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# What Logistic Regression leans on
ohe = logreg.named_steps["pre"].named_transformers_["cat"]
cat_names = list(ohe.get_feature_names_out(CAT_FEATS))
all_names = NUM_FEATS + cat_names
coefs = logreg.named_steps["clf"].coef_[0]
coef_s = pd.Series(coefs, index=all_names).sort_values(key=abs, ascending=False)
print("Top 8 Logistic Regression coefficients (by |value|):")
print(coef_s.head(8))
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Top 8 Logistic Regression coefficients (by |value|):
log_impressions_90d          1.498923
impression_tier_low          0.845170
position_tier_top_3         -0.795715
impression_tier_excellent   -0.615521
word_count_tier_1000-2000    0.596045
log_clicks_90d              -0.571222
word_count                   0.515985
freshness_tier_31-90        -0.467062
dtype: float64


**Top features make sense, and one is a useful warning sign.** `log_impressions_90d` is the
single strongest driver (positive) — busier pages are more likely flagged as declining, which
lines up with the Week-4 audit finding that decline concentrates in the moderate/good traffic
bands, not the extremes. `position_tier_top_3` (negative) and `impression_tier_excellent`
(negative) both push *down* — already-winning pages are the least likely to be "declining,"
which is intuitive. None of these are suspiciously perfect (no single feature dominates near
1.0 probability by itself), which is the leakage sanity check the training skill asks for.

In [6]:
# Where is the model wrong? Look at the biggest misses on the test set.
test_df = df.iloc[test_idx].copy()
test_df["pred_proba"] = logreg_proba
test_df = test_df.sort_values("pred_proba", ascending=False)

review_cols = ["content_id", "pred_proba", "impressions_90d", "avg_position", "ctr",
               "impression_tier", "freshness_tier", "content_age_days", "is_declining_label"]

print("=== False positives: high predicted probability, but NOT declining ===")
print(test_df[test_df["is_declining_label"] == 0].head(3)[review_cols].to_string(index=False))

print("\n=== False negatives: low predicted probability, but IS declining ===")
print(test_df[test_df["is_declining_label"] == 1].tail(3)[review_cols].to_string(index=False))

=== False positives: high predicted probability, but NOT declining ===
          content_id  pred_proba  impressions_90d  avg_position  ctr impression_tier freshness_tier  content_age_days  is_declining_label
content_7be5f150dc65    0.966728              290           5.9 0.00             low           0-30                96                   0
content_41baf0722ad9    0.945409             3115          12.8 0.00            good         91-180               275                   0
content_374e795aab68    0.945257              235          31.0 0.85             low           0-30               181                   0

=== False negatives: low predicted probability, but IS declining ===
          content_id  pred_proba  impressions_90d  avg_position  ctr impression_tier freshness_tier  content_age_days  is_declining_label
content_c268b1716236    0.078434                3          41.7  0.0             low           0-30               502                   1
content_e18144cbd19d    0.06497

## Self-check

Before you submit, confirm each line honestly:

- [✅ ] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [ ✅] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.